---
## Step 1 — Install and load the chatbot


In [1]:
!pip install transformers gradio -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL = 'facebook/blenderbot-400M-distill'

print('Loading chatbot model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
print('Chatbot ready!')


Loading chatbot model...


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/62.9k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  730MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

Chatbot ready!


---
## Step 2 — Create the chat function

> We store the conversation as a simple list.
> Every message (yours + bot) gets added to that list.
> The bot reads the whole list each time — that is its memory!


In [2]:
def chat(user_message, history):
    '''
    user_message : what you typed
    history      : list of all messages so far
    '''
    # Add your message to history
    history.append({'role': 'user', 'content': user_message})

    # Build the full conversation text for the model
    # Each message separated by a special token
    conversation = tokenizer.sep_token.join(
        [m['content'] for m in history]
    )

    # Tokenize and generate reply
    inputs = tokenizer(
        conversation,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 128
    )
    output = model.generate(
        **inputs,
        max_new_tokens = 60
    )
    bot_reply = tokenizer.decode(output[0], skip_special_tokens=True)

    # Add bot reply to history
    history.append({'role': 'bot', 'content': bot_reply})

    return bot_reply, history

print('chat() function ready!')


chat() function ready!


---
## Step 3 — Send one message


In [3]:
# Start with empty history
history = []

# Send first message
reply, history = chat('What is artificial intelligence?', history)

print('You :', 'What is artificial intelligence?')
print('Bot :', reply)


You : What is artificial intelligence?
Bot :  It is the study of human behavior and how it affects the physical, mental, social, and intellectual development of a person.


---
## Step 4 — See the memory in action!

> Tell the bot something personal.
> Then ask a follow-up — it should remember!


In [4]:
# Fresh start
history = []

# Message 1 — tell it something
reply1, history = chat('My favourite sport is cricket.', history)
print('You : My favourite sport is cricket.')
print('Bot :', reply1)
print()

# Message 2 — ask a follow-up
reply2, history = chat('What sport did I just mention?', history)
print('You : What sport did I just mention?')
print('Bot :', reply2)
print()
print('The bot should remember cricket!')


You : My favourite sport is cricket.
Bot :  Cricket is a great sport. It is played on a rectangular field.

You : What sport did I just mention?
Bot :  It is a team sport that involves hitting a ball with a stick to score points.

The bot should remember cricket!


---
## Step 5 — Have a full conversation


In [5]:
# Fresh start
history = []

questions = [
    'Tell me about smart cities.',
    'What sensors are used in smart cities?',
    'How does AI help manage traffic?',
    'What are the benefits for citizens?',
]

print('Full conversation about Smart Cities:')
print('=' * 45)
print()

for i, question in enumerate(questions, 1):
    reply, history = chat(question, history)
    print(f'Turn {i}')
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


Full conversation about Smart Cities:

Turn 1
You : Tell me about smart cities.
Bot :  Smaller cities are the best. Small towns are so much more peaceful and quiet.

Turn 2
You : What sensors are used in smart cities?
Bot :  I live in a small town and it is very quiet and peaceful. I love it.

Turn 3
You : How does AI help manage traffic?
Bot :  I'm not sure, but I do know that there are more than seventy million people in the US that live in small towns.

Turn 4
You : What are the benefits for citizens?
Bot :  It depends on the size of the city and the population density of the area.



---
## Step 6 — Try your own topic!

> ✏️ Change the questions to anything you like!


In [6]:
# ✏️ Change these questions!
history = []

my_questions = [
    'Tell me about machine learning.',
    'How is it different from regular programming?',
    'Can you give me a simple example?',
]

for i, question in enumerate(my_questions, 1):
    reply, history = chat(question, history)
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


You : Tell me about machine learning.
Bot :  Well, it's a form of artificial intelligence. It's used in many fields of science and technology.

You : How is it different from regular programming?
Bot :  Well, I'm not sure how it's different, but I know it involves a lot of math.

You : Can you give me a simple example?
Bot :  It involves many different things, such as algebra, geometry, and calculus.



---
## Step 7 — Build a Gradio Chatbot App

> Type any message — bot replies and remembers!
> Click **Clear** to start a fresh conversation.


In [7]:
import gradio as gr

# Conversation memory for the BlenderBot model
memory = {"history": []}

def respond(user_message):
    if not user_message or not user_message.strip():
        return "", "Please enter a message first."

    try:
        # Use the existing chat() function from Step 2.
        reply, memory["history"] = chat(user_message.strip(), memory["history"])
        return reply, ""
    except Exception as exc:
        return "", f"Sorry, the chatbot could not respond: {exc}"

def clear_chat():
    memory["history"] = []
    return "", "", "Conversation cleared. Start a new chat!"

def flag_response(user_message, bot_reply):
    if not user_message.strip() or not bot_reply.strip():
        return "Send a message first before flagging a response."
    # Demo only: this confirmation is not sent to a server.
    return "Feedback flagged locally for this demo. No report was sent."

def load_example(example):
    return example, "", ""

custom_css = """
.gradio-container {
    background: #101010 !important;
    color: #f5f5f5 !important;
    max-width: 100% !important;
    padding: 16px 3.7% 36px !important;
}
#page-title {
    text-align: center;
    font-size: 32px;
    font-weight: 700;
    margin: 0 0 20px 0;
}
#subtitle { margin-bottom: 24px; font-size: 16px; }
.panel {
    background: #272729 !important;
    border: 1px solid #414149 !important;
    border-radius: 5px !important;
    padding: 14px !important;
}
.panel label { color: #f2f2f2 !important; }
textarea {
    background: #252528 !important;
    color: #f5f5f5 !important;
    border: 1px solid #414149 !important;
    border-radius: 5px !important;
    min-height: 145px !important;
}
.action-btn {
    min-height: 50px !important;
    font-size: 17px !important;
    font-weight: 600 !important;
    border-radius: 8px !important;
}
#submit-btn {
    background: #f4510b !important;
    color: white !important;
    border: none !important;
}
#clear-btn, #flag-btn {
    background: #55555e !important;
    color: white !important;
    border: none !important;
}
.example-btn {
    background: transparent !important;
    border: 1px solid #414149 !important;
    color: #f4f4f4 !important;
    border-radius: 8px !important;
}
"""

examples = [
    "What is artificial intelligence?",
    "Tell me about Python programming.",
    "What are the benefits of the internet?",
    "How can I become a full-stack developer?"
]

with gr.Blocks(
    title="AI Chatbot",
    theme=gr.themes.Base(),
    css=custom_css
) as app:
    gr.Markdown("# AI Chatbot", elem_id="page-title")
    gr.Markdown(
        "Ask questions and chat with AI. The chatbot keeps conversation memory until you clear it.",
        elem_id="subtitle"
    )

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, elem_classes=["panel"]):
            user_input = gr.Textbox(
                label="Your Message",
                placeholder="Type your message here...",
                lines=5,
                max_lines=8
            )
        with gr.Column(scale=1, elem_classes=["panel"]):
            bot_output = gr.Textbox(
                label="Chatbot Response",
                placeholder="The chatbot's response will appear here...",
                lines=5,
                max_lines=8,
                interactive=False
            )

    with gr.Row():
        clear_btn = gr.Button(
            "Clear", elem_id="clear-btn", elem_classes=["action-btn"], scale=1
        )
        submit_btn = gr.Button(
            "Submit", elem_id="submit-btn", elem_classes=["action-btn"], scale=1
        )
        flag_btn = gr.Button(
            "Flag", elem_id="flag-btn", elem_classes=["action-btn"], scale=2
        )

    status = gr.Markdown("")
    gr.Markdown("☷  Examples")

    with gr.Row():
        ex1 = gr.Button(examples[0], elem_classes=["example-btn"])
        ex2 = gr.Button(examples[1], elem_classes=["example-btn"])
        ex3 = gr.Button(examples[2], elem_classes=["example-btn"])
    with gr.Row():
        ex4 = gr.Button(examples[3], elem_classes=["example-btn"])

    submit_btn.click(
        fn=respond,
        inputs=[user_input],
        outputs=[bot_output, status]
    )
    user_input.submit(
        fn=respond,
        inputs=[user_input],
        outputs=[bot_output, status]
    )
    clear_btn.click(
        fn=clear_chat,
        outputs=[user_input, bot_output, status]
    )
    flag_btn.click(
        fn=flag_response,
        inputs=[user_input, bot_output],
        outputs=[status]
    )

    for button, example in [
        (ex1, examples[0]),
        (ex2, examples[1]),
        (ex3, examples[2]),
        (ex4, examples[3])
    ]:
        button.click(
            fn=lambda e=example: (e, "", ""),
            outputs=[user_input, bot_output, status]
        )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://63816cded46dd7eb25.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
app.close()
print('App closed.')


Closing server running on port: 7860
App closed.


---
## What we learned

| Step | What we did |
|------|-------------|
| 1 | Loaded Blenderbot model directly |
| 2 | Created chat() function with history list |
| 3 | Sent one message and got a reply |
| 4 | Saw memory in action |
| 5 | Had a full 4-turn conversation |
| 6 | Tried our own topic |
| 7 | Built Gradio chatbot app |

### Key words:
- **`history`** — a list that stores all messages
- **Memory** — passing the same history list to every message
- **Clear** — resetting history to empty list `[]`

**Tomorrow — Day 9:** Translation + Summarization!
